# Reuters-10K and Fashion-MNIST
This notebook prepares Reuters features and runs the same clustering models on either dataset. Fashion-MNIST is the default; Reuters preparation is optional because the raw corpus is not included.

The historical Reuters document IDs, vocabulary and preprocessing snapshot were not retained. The two preparation routes below are explicit reconstructions, so their results need to be reported with the chosen route.

In [ ]:
%run ./functions.ipynb
import gzip
import json
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.preprocessing import MinMaxScaler, normalize

seed = 42
data_dir = Path('data')
dataset_names = ['fashion_mnist']  # add 'reuters10k' after preparing it
n_samples = 2000                   # None: full 70,000 images / prepared 10,000 texts
pretrain_epochs = 3
joint_epochs = 10
hidden = [128, 64]
latent_dim = 10
batch_size = 256
repeats = 1

reuters_preparation = None         # None, 'raw_tokens', or 'sklearn_variant'
raw_reuters_dir = data_dir / 'rcv1_raw'
reuters_path = data_dir / 'reuters10k.npz'
run_two_stage = True
run_joint = True
run_convolutional = False          # image datasets only
run_padded_text_conv = False       # optional reconstructed Reuters convolution

## Reuters input
For the raw route, place `rcv1-v2.topics.qrels` and the `.dat` / `.dat.gz` stemmed token files in `data/rcv1_raw/`. The four labels are CCAT, GCAT, MCAT and ECAT. Documents with more than one of these labels are removed. Vocabulary and IDF are fitted on the eligible corpus before 10,000 documents are sampled.

The raw route can need several GB of memory while building the vocabulary. The sklearn route uses already computed RCV1 TF-IDF vectors and selects features by document frequency; it is a different preprocessing variant. Set `reuters_preparation` explicitly to run either route.

Prepared files use CSR arrays (`data`, `indices`, `indptr`, `shape`) plus `y`, or dense arrays `X` and `y`. Extra metadata does not affect the shared loader.

In [ ]:
def save_reuters_npz(X, y, path, metadata=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    X = sparse.csr_matrix(X, dtype=np.float32)
    np.savez_compressed(path, data=X.data, indices=X.indices, indptr=X.indptr,
                        shape=np.array(X.shape), y=np.asarray(y, dtype=np.int64),
                        metadata=np.array(json.dumps(metadata or {})))


def iter_rcv1_tokens(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8', errors='replace') as stream:
        doc_id, words = None, []
        for line in stream:
            if line.startswith('.I '):
                if doc_id is not None:
                    yield doc_id, ' '.join(words)
                doc_id, words = int(line.split()[1]), []
            elif line.strip() != '.W':
                words.append(line.strip())
        if doc_id is not None:
            yield doc_id, ' '.join(words)


def prepare_reuters_from_tokens(raw_dir, output='data/reuters10k.npz',
                                n_samples=10000, n_features=2000, seed=42):
    raw_dir = Path(raw_dir)
    roots = ['CCAT', 'GCAT', 'MCAT', 'ECAT']
    memberships = {}
    with (raw_dir / 'rcv1-v2.topics.qrels').open() as stream:
        for line in stream:
            topic, doc_id, relevance = line.split()[:3]
            if topic in roots and int(relevance) > 0:
                memberships.setdefault(int(doc_id), set()).add(roots.index(topic))
    single = {i: next(iter(labels)) for i, labels in memberships.items() if len(labels) == 1}
    files = sorted(p for p in raw_dir.iterdir() if p.name.endswith(('.dat', '.dat.gz')))
    if not files:
        raise FileNotFoundError('No RCV1 token .dat or .dat.gz files were found.')
    documents, labels, ids, seen = [], [], [], set()
    for file in files:
        for doc_id, text in iter_rcv1_tokens(file):
            if doc_id in single and doc_id not in seen:
                documents.append(text)
                labels.append(single[doc_id])
                ids.append(doc_id)
                seen.add(doc_id)
    if len(documents) < n_samples:
        raise ValueError(f'Found {len(documents)} single-root documents, need {n_samples}.')
    vectorizer = CountVectorizer(max_features=n_features, tokenizer=str.split,
                                 token_pattern=None, lowercase=False, dtype=np.float32)
    counts = vectorizer.fit_transform(documents)
    X = TfidfTransformer(norm='l2', sublinear_tf=True).fit_transform(counts).tocsr()
    selected = np.random.default_rng(seed).choice(len(documents), n_samples, replace=False)
    X, y = X[selected], np.asarray(labels)[selected]
    metadata = {'route': 'raw_rcv1_tokens', 'root_order': roots, 'seed': seed,
                'eligible_documents': len(documents), 'sublinear_tf': True, 'norm': 'l2',
                'document_ids': np.asarray(ids)[selected].tolist(),
                'vocabulary': vectorizer.get_feature_names_out().tolist()}
    save_reuters_npz(X, y, output, metadata)
    return X, y, metadata


def prepare_reuters_from_sklearn(output='data/reuters10k.npz', data_home='data/rcv1',
                                 n_samples=10000, n_features=2000, seed=42):
    from sklearn.datasets import fetch_rcv1
    reuters = fetch_rcv1(data_home=str(data_home), shuffle=False)
    roots = ['CCAT', 'GCAT', 'MCAT', 'ECAT']
    columns = [list(reuters.target_names).index(name) for name in roots]
    targets = reuters.target[:, columns].toarray()
    eligible = np.flatnonzero((targets > 0).sum(axis=1) == 1)
    if len(eligible) < n_samples:
        raise ValueError('Not enough single-root documents.')
    source = reuters.data[eligible].tocsr()
    document_frequency = np.asarray((source > 0).sum(axis=0)).ravel()
    features = np.argsort(-document_frequency, kind='stable')[:n_features]
    selected = np.random.default_rng(seed).choice(len(eligible), n_samples, replace=False)
    X = normalize(source[selected][:, features]).astype(np.float32)
    y = targets[eligible[selected]].argmax(axis=1)
    metadata = {'route': 'sklearn_rcv1_tfidf_variant', 'root_order': roots, 'seed': seed,
                'feature_indices': features.tolist(),
                'document_ids': reuters.sample_id[eligible[selected]].tolist()}
    save_reuters_npz(X, y, output, metadata)
    return X, y, metadata

In [ ]:
if reuters_preparation == 'raw_tokens':
    reuters_X, reuters_y, reuters_metadata = prepare_reuters_from_tokens(
        raw_reuters_dir, reuters_path, seed=seed)
    print(reuters_X.shape, reuters_metadata['route'])
elif reuters_preparation == 'sklearn_variant':
    reuters_X, reuters_y, reuters_metadata = prepare_reuters_from_sklearn(
        reuters_path, data_dir / 'rcv1', seed=seed)
    print(reuters_X.shape, reuters_metadata['route'])
elif reuters_preparation is not None:
    raise ValueError('Choose None, raw_tokens or sklearn_variant.')

## Optional padded text convolution
This preserves the earlier GitHub implementation for the Reuters convolution comparison. Pad the feature vector with zeros to a square whose side is a multiple of four, apply a convolutional VAE, then crop the decoded vector back to the original feature count. Reconstruction uses MSE and a linear output layer.

Feature adjacency in this square is arbitrary and depends on vocabulary order. This is a reconstructed experiment, not a recovered historical preprocessing rule or an assertion that text features have a natural image layout. It is off by default (`run_padded_text_conv=False`).

In [ ]:
class PaddedTextVAE(nn.Module):
    def __init__(self, input_dim=2000, latent_dim=10, dropout=0.2):
        super().__init__()
        if input_dim < 1:
            raise ValueError('input_dim must be positive.')
        self.input_dim = input_dim
        self.hidden_dimension = latent_dim
        self.side = 4 * int(np.ceil(np.sqrt(input_dim) / 4))
        self.small = self.side // 4
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Dropout2d(dropout),
            ResidualBlock(64), ResidualBlock(64), nn.Flatten())
        self.fc_stats = nn.Linear(64 * self.small ** 2, 2 * latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 64 * self.small ** 2)
        self.decoder = nn.Sequential(
            ResidualBlock(64), ResidualBlock(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1))
        self.apply(initialise_weights)

    def encode_stats(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError('Expected a [samples, input_dim] feature matrix.')
        padded = F.pad(x, (0, self.side ** 2 - self.input_dim))
        image = padded.reshape(-1, 1, self.side, self.side)
        mu, log_var = self.fc_stats(self.encoder(image)).chunk(2, dim=1)
        return mu, log_var.clamp(-20, 10)

    def encode(self, x):
        return self.encode_stats(x)[0]

    def decode(self, z):
        hidden = self.fc_decode(z).reshape(-1, 64, self.small, self.small)
        return self.decoder(hidden).flatten(1)[:, :self.input_dim]

    def forward(self, x):
        mu, log_var = self.encode_stats(x)
        z = VariationalAutoEncoder.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

## Compare clustering methods
The defaults below run a small comparison. For the larger experiment, use `n_samples=None`, `hidden=[500, 500, 2000]` and a longer training budget. Every repeat starts from its own seed.

The fully connected model works directly on Reuters TF-IDF features with MSE reconstruction. `run_convolutional` enables the 28 x 28 image model. `run_padded_text_conv` separately enables the reconstructed Reuters variant above; its feature layout is a modeling assumption.

Other baselines are in `deep_clustering.ipynb` and `spectralnet.ipynb`; set their dataset to `reuters10k` and cluster count to 4 to use this prepared file.

In [ ]:
rows = []
runs = {}
for dataset_name in dataset_names:
    X, y = load_data(dataset_name, n_samples=n_samples, seed=seed, data_dir=data_dir)
    n_clusters = 4 if dataset_name == 'reuters10k' else 10
    loss = 'mse' if dataset_name == 'reuters10k' else 'bce'
    dimensions = [X.shape[1], *hidden, latent_dim]
    for run_seed in range(seed, seed + repeats):
        seed_everything(run_seed)
        array = X.numpy()
        predictions = {
            'KMeans': KMeans(n_clusters, n_init=10, random_state=run_seed).fit_predict(array),
            'GMM': GaussianMixture(n_clusters, covariance_type='diag',
                                   random_state=run_seed).fit_predict(array),
            'NMF': NMF(n_clusters, init='random', max_iter=200,
                       random_state=run_seed).fit_transform(array).argmax(1)
        }
        if run_two_stage:
            for kind, model_class in [('AE', AutoEncoder), ('VAE', VariationalAutoEncoder)]:
                seed_everything(run_seed)
                model = model_class(dimensions, final_activation=(
                    nn.Identity() if loss == 'mse' else nn.Sigmoid())).to(device)
                pre_train(X, model, epochs=pretrain_epochs, batch_size=batch_size,
                          loss=loss, seed=run_seed)
                Z = vae_encode(X, model, batch_size)
                predictions[kind + '+KMeans'] = KMeans(
                    n_clusters, n_init=10, random_state=run_seed).fit_predict(Z)
                nonnegative_Z = np.maximum(MinMaxScaler().fit_transform(Z), 0)
                predictions[kind + '+NMF'] = NMF(
                    n_clusters, init='random', max_iter=200,
                    random_state=run_seed).fit_transform(nonnegative_Z).argmax(1)
        if run_joint:
            joint = train_vae_mf_model_optimized(
                X, n_clusters, dimensions, pretrain_epochs=pretrain_epochs,
                epochs=joint_epochs, batch_size=batch_size, loss=loss, seed=run_seed, y=y)
            predictions['Joint VAE+MF'] = joint['pred']
            runs[(dataset_name, run_seed, 'fc')] = joint
        if run_convolutional and dataset_name in ['mnist', 'fashion_mnist']:
            seed_everything(run_seed)
            model = ConvolutionalVAE(input_dim=X.shape[1], latent_dim=latent_dim).to(device)
            joint = train_vae_mf_model_optimized(
                X, n_clusters, model=model, pretrain_epochs=pretrain_epochs,
                epochs=joint_epochs, batch_size=batch_size, loss=loss, seed=run_seed, y=y)
            predictions['Joint ConvVAE+MF'] = joint['pred']
            runs[(dataset_name, run_seed, 'conv')] = joint
        if run_padded_text_conv and dataset_name == 'reuters10k':
            seed_everything(run_seed)
            model = PaddedTextVAE(X.shape[1], latent_dim).to(device)
            joint = train_vae_mf_model_optimized(
                X, n_clusters, model=model, pretrain_epochs=pretrain_epochs,
                epochs=joint_epochs, batch_size=batch_size, loss='mse', seed=run_seed, y=y)
            predictions['Padded text ConvVAE (reconstructed)'] = joint['pred']
            runs[(dataset_name, run_seed, 'padded_text_conv')] = joint
        for method, pred in predictions.items():
            rows.append({'dataset': dataset_name, 'method': method, 'seed': run_seed,
                         'n_samples': len(X), **score_dict(y, pred)})
scores = pd.DataFrame(rows)
display(scores)
scores.groupby(['dataset', 'method'])[['ACC', 'NMI', 'ARI', 'F1', 'Purity']].agg(['mean', 'std'])

In [ ]:
save_results = False
if save_results:
    Path('results').mkdir(exist_ok=True)
    scores.to_csv('results/reuters_fashion_scores.csv', index=False)